# Data Storytelling Project Notebook

Brandon Kung 

Things to do:
- Redo the extraction of the data so you filter out non-kinetic missions, those with missing geographic coordinates, and those that INTERSECT with the Laotian border
- Revise the comments
- Remove "National" observation from the subnational HDI dataset

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

%matplotlib inline
%config InlineBackend.figure_format = "retina"

# Loading in the Subnational HDI Dataset and Exploration 

In [ ]:
shdi_df = pd.read_csv('../data/GDL-Subnational-HDI-data.csv')

In [ ]:
shdi_df.head()

In [ ]:
shdi_df.info()

The Lao data has more subnational units and it was bombed more, so we will focus on Lao.

In [ ]:
# filtering dataset to include only Laotian provinces
lao_shdi = shdi_df[(shdi_df['ISO_Code'] == 'LAO') & (shdi_df['Level'] != 'National')]
lao_shdi['Level'].value_counts()

In [ ]:
id_cols = ['Country', 'Continent', 'ISO_Code', 'Level', 'GDLCODE', 'Region']

lao_shdi_df = pd.melt(
    lao_shdi,
    id_vars=id_cols,
    var_name='year',
    value_name='shdi_value'  
)

In [ ]:
print(lao_shdi_df.columns)

In [ ]:
lao_shdi_df.head(20)

It seems like the ISO Code is not mapped by subnational divisions, so let's fix that.

In [ ]:
# Dictionary mapping Employee to Department
ISO_dict = {'Attapeu': 'LA-AT', 'Bokeo': 'LA-BK', 'Borikhamxay': 'LA-BL', 'Champasack': 'LA-CH', 
            'Huaphanh': 'LA-HO', 'Khammuane': 'LA-KH', 'Luangnamtha': 'LA-LM', 'Luangprabang': 'LA-LP', 'Oudomxay': 'LA-OU', 'Phongsaly': 'LA-PH', 
            'Saravane': 'LA-SL', 'Savannakhet': 'LA-SV', 'Sayabury': 'LA-XA', 'Sekong': 'LA-XE', 'Vientiane Municipality': 'LA-VT', 
           'Vientiane Province': 'LA-VI', 'Xiengkhuang': 'LA-XI'}

# Add the new column
lao_shdi_df['ISO_Code'] = lao_shdi_df['Region'].map(ISO_dict)
lao_shdi_df['ISO_Code'].head(17)

In [ ]:
# Graphing line trend for exploration purposes
plt.figure(figsize=(20, 6))

for key, grp in lao_shdi_df.groupby(['GDLCODE']):
    plt.plot(grp['year'], grp['shdi_value'], alpha=0.05, color='blue')

plt.title('Trajectories of Subnational HDI Over Time')
plt.xlabel('Year')
plt.ylabel('Subnational HDI Score')

# Loading in the THOR Bombing Dataset

In [ ]:
# import polars as pl

In [ ]:
# using polars to filter the data because it's very large 
#columns_to_keep = [
    #'TGTCOUNTRY',
    #'TGTLATDD_DDD_WGS84',
    #'TGTLONDDD_DDD_WGS84',
    #'MSNDATE',
    #'WEAPONTYPEWEIGHT',
    #'NUMWEAPONSDELIVERED',
    #'MFUNC_DESC_CLASS',
    #'WEAPONTYPECLASS',
    #'WEAPONTYPE'
#]

#(
    #pl.scan_csv('../data/large_thor_file.csv', ignore_errors=True)
    #.select(columns_to_keep)
    #.filter(pl.col('TGTCOUNTRY').str.to_uppercase() == 'LAOS')
    #.collect()
    #.write_csv('laos_bombing_clean.csv')
#)

In [ ]:
laos_df = pd.read_csv('../data/laos_bombing_clean.csv')

In [ ]:
laos_df.head()

In [ ]:
print(laos_df['MFUNC_DESC_CLASS'].value_counts())

In [ ]:
laos_df_clean = laos_df[laos_df['MFUNC_DESC_CLASS'].isin(['KINETIC'])]
laos_df_clean.head()

In [ ]:
laos_df_clean = laos_df_clean.dropna(subset=['TGTLATDD_DDD_WGS84', 'TGTLONDDD_DDD_WGS84'])
laos_df_clean.isna().sum()

In [ ]:
laos_df_clean

# Integrating Level 1 Boundaries (Provinces)

In [78]:
# Loading the administrative boundaries
gdf_provinces = gpd.read_file('../data/LaosBoundaryLvl1.json')
print(gdf_provinces.head())

     GID_1 GID_0 COUNTRY       NAME_1                         VARNAME_1  \
0  LAO.1_1   LAO    Laos       Attapu  Attopu|Atpu|Attapeu|Attopeu|Muan   
1  LAO.2_1   LAO    Laos        Bokeo                                NA   
2  LAO.3_1   LAO    Laos  Bolikhamxai  Bolikhamsai|Bolikhamxay|Borikham   
3  LAO.4_1   LAO    Laos    Champasak  Bassac|Champassack|Champassak|Ch   
4  LAO.5_1   LAO    Laos     Houaphan     HuaPhan|Huaphanh|SamNeua|XamN   

  NL_NAME_1   TYPE_1 ENGTYPE_1 CC_1 HASC_1  ISO_1  \
0        NA  Khoueng  Province   NA  LA.AT  LA-AT   
1        NA  Khoueng  Province   NA  LA.BK  LA-BK   
2        NA  Khoueng  Province   NA  LA.BL  LA-BL   
3        NA  Khoueng  Province   NA  LA.CH  LA-CH   
4        NA  Khoueng  Province   NA  LA.HO     NA   

                                            geometry  
0  MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...  
1  MULTIPOLYGON (((100.7467 19.962, 100.7339 19.9...  
2  MULTIPOLYGON (((104.0322 18.8128, 104.0392 18....  
3  MULTIPO

In [ ]:
# Setting up the plotting canvas
fig, ax = plt.subplots(figsize=(10, 10))

# Plotting the GeoDataFrame
gdf_provinces.plot(
    ax=ax, 
    facecolor='whitesmoke', 
    edgecolor='black', 
    linewidth=0.8
)

# Formatting the map
plt.title('Laos Administrative Boundaries', fontsize=16, pad=20)
ax.axis('off')

# 6. Render the plot
plt.tight_layout()
plt.show()

# Merging the datasets

In [81]:
# Merging the HDI and administrative boundaries 
gdf_provinces.rename(columns={'ISO_1': 'ISO_Code'}, inplace=True)
shdi_lvl1_merge = pd.merge(gdf_provinces, lao_shdi_df, on='ISO_Code', how='left')
shdi_lvl1_merge.head()

,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_Code,geometry,Country,Continent,Level,GDLCODE,Region,year,shdi_value
0,LAO.1_1,LAO,Laos,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,NA,Khoueng,Province,NA,LA.AT,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",Lao,Asia/Pacific,Subnat,LAOr117,Attapeu,1990,0.379
1,LAO.1_1,LAO,Laos,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,NA,Khoueng,Province,NA,LA.AT,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",Lao,Asia/Pacific,Subnat,LAOr117,Attapeu,1991,0.384
2,LAO.1_1,LAO,Laos,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,NA,Khoueng,Province,NA,LA.AT,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",Lao,Asia/Pacific,Subnat,LAOr117,Attapeu,1992,0.389
3,LAO.1_1,LAO,Laos,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,NA,Khoueng,Province,NA,LA.AT,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",Lao,Asia/Pacific,Subnat,LAOr117,Attapeu,1993,0.395
4,LAO.1_1,LAO,Laos,Attapu,Attopu|Atpu|Attapeu|Attopeu|Muan,NA,Khoueng,Province,NA,LA.AT,LA-AT,"MULTIPOLYGON (((107.1091 14.394, 107.1061 14.3...",Lao,Asia/Pacific,Subnat,LAOr117,Attapeu,1994,0.403


In [ ]:
# Convert the pandas DataFrame into a GeoDataFrame of points
# WGS84 (EPSG:4326) is the standard CRS for latitude/longitude coordinates
bombing_gdf = gpd.GeoDataFrame(
    laos_df_clean, 
    geometry=gpd.points_from_xy(laos_df_clean['TGTLONDDD_DDD_WGS84'], laos_df_clean['TGTLATDD_DDD_WGS84']),
    crs="EPSG:4326"
)

# 3. Align CRS
if bombing_gdf.crs != gdf_provinces.crs:
    gdf_provinces = gdf_provinces.to_crs(bombing_gdf.crs)

# 4. Perform the Spatial Join
# 'predicate="within"' ensures we find which province polygon each point sits inside
# 'how="left"' keeps all points and appends the province columns to them
points_with_provinces = gpd.sjoin(bombing_gdf, gdf_provinces, how='left', predicate='within')

# 5. Group by Province and Aggregate the Impact
# Replace 'NAME_1' with the actual column name for province names in your shapefile
spatial_summary = points_with_provinces.groupby('NAME_1').agg(
    total_missions=('TGTCOUNTRY', 'size'),
    total_tonnage=('WEAPONTYPEWEIGHT', 'sum')
).reset_index()

# View the result
print(spatial_summary.head())

In [ ]:
# Merge the aggregated statistics back to the geometry data
bombing_admin_df = gdf_provinces.merge(spatial_summary, on='NAME_1', how='left')

# Fill provinces that had zero bombing missions with 0 instead of NaN
bombing_admin_df['total_tonnage'] = bombing_admin_df['total_tonnage'].fillna(0)
bombing_admin_df['total_missions'] = bombing_admin_df['tota_missions'].fillna(0)

# Integrating District Boundaries

In [ ]:
# Loading district boundaries
gdf_districts = gpd.read_file('../data/LaosBoundaryLvl2.json')
print(gdf_districts.head())

# 3. Set up the plotting canvas
fig, ax = plt.subplots(figsize=(10, 10))

# 4. Plot the GeoDataFrame
# - facecolor: fills the polygons. 'whitesmoke' gives a nice base map feel.
# - edgecolor: color of the province borders.
# - linewidth: thickness of the borders.
gdf_districts.plot(
    ax=ax, 
    facecolor='whitesmoke', 
    edgecolor='black', 
    linewidth=0.8
)

# 5. Formatting the map
plt.title('Laos Administrative Boundaries', fontsize=16, pad=20)

# Turn off the latitude/longitude axis labels for a cleaner, map-like appearance
ax.axis('off')

# 6. Render the plot
plt.tight_layout()
plt.show()